In [9]:
import torch
import warnings
import subprocess
import os
import shutil
import sys

# Suppress expected warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Suppress expected warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def convert_torchscript_to_onnx(torchscript_path, output_path):
    """Convert TorchScript model to ONNX with modern approach"""
    try:
        print(f"Loading TorchScript model from {torchscript_path}")
        model = torch.jit.load(torchscript_path, map_location='cpu')
        model.eval()
        dummy_input = torch.randn(1, 3, 640, 640)
        try:
            print("Attempting export with dynamo=True (new method)...")
            torch.onnx.export(
                model, dummy_input, output_path,
                export_params=True, opset_version=11,
                do_constant_folding=True,
                input_names=['input'], output_names=['output'],
                dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
                dynamo=True
            )
            print(f"✅ Exported with dynamo: {output_path}")
            return True
        except Exception as e:
            print(f"Dynamo export failed: {e}\nFalling back to legacy export...")
            torch.onnx.export(
                model, dummy_input, output_path,
                export_params=True, opset_version=11,
                do_constant_folding=True,
                input_names=['input'], output_names=['output'],
                dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
            )
            print(f"✅ Exported legacy: {output_path}")
            return True
    except Exception as e:
        print(f"❌ Failed TorchScript→ONNX: {e}")
        return False

def convert_openvino_to_onnx(openvino_xml_path, output_path):
    """Convert OpenVINO IR to ONNX using Model Optimizer on any OS."""
    try:
        # Ensure openvino-dev[onnx] is installed
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'openvino-dev[onnx]'],
                       check=True, stdout=subprocess.DEVNULL)

        # Use Python module entrypoint instead of mo_onnx script
        cmd = [
            sys.executable, '-m', 'openvino.tools.mo_onnx',
            '--input_model', openvino_xml_path,
            '--output', output_path,
            '--input_shape', '[1,3,640,640]'
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"✅ Converted OpenVINO→ONNX: {output_path}")
            return True
        else:
            print(f"❌ Model Optimizer failed:\n{result.stdout}\n{result.stderr}")
            return False
    except Exception as e:
        print(f"❌ Failed OpenVINO→ONNX: {e}")
        return False

if __name__ == "__main__":
    conversions = []

    # TorchScript → ONNX
    # ts_path = "models/best.torchscript"
    # ts_onnx = "models/best_from_torchscript.onnx"
    # if os.path.exists(ts_path):
    #     success = convert_torchscript_to_onnx(ts_path, ts_onnx)
    #     conversions.append(("TorchScript", success))

    # OpenVINO → ONNX
    ov_xml = "models/best_openvino_model/best.xml"
    ov_onnx = "models/best_from_openvino.onnx"
    if os.path.exists(ov_xml):
        success = convert_openvino_to_onnx(ov_xml, ov_onnx)
        conversions.append(("OpenVINO", success))

    # Original ONNX
    # orig_onnx = "models/best.onnx"
    # orig_copy = "models/best_original.onnx"
    # if os.path.exists(orig_onnx):
    #     shutil.copy(orig_onnx, orig_copy)
    #     conversions.append(("Original ONNX", True))

    print("\n" + "="*50)
    print("CONVERSION SUMMARY:")
    for model_type, success in conversions:
        status = "✅ SUCCESS" if success else "❌ FAILED"
        print(f"{model_type:15}: {status}")

    print("\nReady ONNX models for Hailo conversion:")
    for model_type, success in conversions:
        if success:
            filename = {
                # "TorchScript": ts_onnx,
                "OpenVINO": ov_onnx
                # "Original ONNX": orig_copy
            }[model_type]
            print(f"  📁 {filename}")


❌ Model Optimizer failed:

C:\Users\ethan\anaconda3\python.exe: No module named openvino.tools.mo_onnx


CONVERSION SUMMARY:
OpenVINO       : ❌ FAILED

Ready ONNX models for Hailo conversion:


In [10]:
import subprocess
import sys
import os
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def convert_openvino_to_onnx(openvino_xml_path, output_path):
    """Convert OpenVINO IR to ONNX using the Model Optimizer (mo_onnx)."""
    try:
        # Install OpenVINO ONNX support if missing
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "openvino-dev[onnx]"],
            check=True, stdout=subprocess.DEVNULL
        )
        # Use the Python module entry point
        cmd = [
            sys.executable, "-m", "openvino.tools.mo_onnx",
            "--input_model", openvino_xml_path,
            "--output", output_path,
            "--input_shape", "[1,3,640,640]"
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"✅ Converted OpenVINO→ONNX: {output_path}")
            return True
        else:
            print("❌ Model Optimizer failed:")
            print(result.stdout)
            print(result.stderr)
            return False
    except Exception as e:
        print(f"❌ Failed OpenVINO→ONNX: {e}")
        return False

if __name__ == "__main__":
    ov_xml = "models/best_openvino_model/best.xml"
    ov_onnx = "models/best_from_openvino.onnx"
    if os.path.exists(ov_xml):
        convert_openvino_to_onnx(ov_xml, ov_onnx)
    else:
        print(f"❌ OpenVINO IR not found at {ov_xml}")


❌ Model Optimizer failed:

C:\Users\ethan\anaconda3\python.exe: No module named openvino.tools.mo_onnx



In [12]:
#!/usr/bin/env python3
import os
import subprocess

# List your ONNX models here
MODELS = {
    "original": "models/best.onnx",
    "from_torchscript": "models/best_from_torchscript.onnx",
    "from_openvino": "models/best_from_openvino.onnx"
}

OUTPUT_DIR = "models/hef_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run(cmd):
    print(f"> {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout, result.stderr)
    return result.returncode == 0

for name, onnx_path in MODELS.items():
    if not os.path.exists(onnx_path):
        print(f"ONNX not found: {onnx_path}")
        continue

    base = os.path.join(".", name)
    har = f"{base}.har"
    qhar = f"{base}.quantized.har"
    hef = f"{OUTPUT_DIR}/{name}.hef"

    print(f"\nConverting {onnx_path} → {hef}")

    # 1. Parse ONNX → HAR
    if not run(["hailo", "parser", "onnx", "--hw-arch", "hailo8l", onnx_path, "--output", har]):
        print(f"Failed parse step for {name}"); continue

    # 2. Optimize & quantize (use real calibration set with --calib-list in production)
    if not run(["hailo", "optimize", har, "--use-random-calib-set", "--output", qhar]):
        print(f"Failed optimize step for {name}"); continue

    # 3. Compile → HEF
    if not run(["hailo", "compile", qhar, "--hw-arch", "hailo8l", "--output", hef]):
        print(f"Failed compile step for {name}"); continue

    print(f"✅ {name}.hef created at {hef}")

print("\nAll conversions done.")



Converting models/best.onnx → models/hef_models/original.hef
> hailo parser onnx --hw-arch hailo8l models/best.onnx --output .\original.har


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [4]:
import torch

# Load your original PyTorch model (not TorchScript)
model = torch.jit.load("models/best.torchscript")
model.eval()

# Dummy input
dummy = torch.randn(1, 3, 640, 640)

# Export with dynamo and dynamic_shapes
torch.export(
    model,
    dummy,
    "best_direct.onnx",
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_shapes={
        "input": {0: "batch", 2: "height", 3: "width"},
        "output": {0: "batch"}
    },
    dynamo=True  # Use new exporter
)

TypeError: 'module' object is not callable